# Free Photo-to-Talking-Video (Hindi audio) — SadTalker

This notebook uses the open-source SadTalker project. It requires a compatible Colab runtime and downloads model checkpoints. Free GPU access is not guaranteed; if no GPU is assigned, switch Runtime → Change runtime type → T4 GPU if available. Never upload private photos/audio you do not have permission to use.

**Input:** one portrait photo + an already generated Hindi audio file. **Output:** talking-photo MP4. This does not translate text to Hindi; create Hindi audio first.

In [ ]:
import os, subprocess, sys
print('Python:', sys.version)
try:
  import torch
  print('CUDA available:', torch.cuda.is_available())
  if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except Exception as e: print('Torch check:', e)
from google.colab import files
uploaded = files.upload()  # Select one portrait image and one Hindi narration audio file
print('Uploaded:', list(uploaded.keys()))

## Install SadTalker
Run the next cell once. This downloads the source and model checkpoints; it may take several minutes.

In [ ]:
%cd /content
!git clone https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
!pip install -q -r requirements.txt
!bash scripts/download_models.sh

In [ ]:
from pathlib import Path
files_now = [p for p in Path('/content').iterdir() if p.is_file()]
print('\n'.join(p.name for p in files_now))
image_path = input('Enter exact uploaded portrait filename (e.g. portrait.jpg): ').strip()
audio_path = input('Enter exact uploaded Hindi audio filename (e.g. hindi.wav): ').strip()
assert Path('/content', image_path).is_file(), 'Photo file not found in /content'
assert Path('/content', audio_path).is_file(), 'Audio file not found in /content'
out_dir = '/content/SadTalker/results/kiran_talking_photo'
Path(out_dir).mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'inference.py', '--driven_audio', str(Path('/content', audio_path)), '--source_image', str(Path('/content', image_path)), '--result_dir', out_dir, '--still', '--preprocess', 'crop']
print('Running SadTalker; this can take several minutes...')
subprocess.run(cmd, cwd='/content/SadTalker', check=True)
videos = sorted(Path(out_dir).rglob('*.mp4'), key=lambda p: p.stat().st_mtime, reverse=True)
videos = [p for p in videos if p.stat().st_size > 1000]
assert videos, 'No MP4 produced. Check the complete inference error above and model downloads.'
output_path = Path('/content/talking_photo_hindi.mp4')
import shutil
shutil.copy2(videos[0], output_path)
print('OUTPUT:', output_path, 'bytes:', output_path.stat().st_size)
from IPython.display import Video, display
display(Video(str(output_path), embed=True))

In [ ]:
from google.colab import files
assert Path('/content/talking_photo_hindi.mp4').is_file() and Path('/content/talking_photo_hindi.mp4').stat().st_size > 1000
files.download('/content/talking_photo_hindi.mp4')

## Troubleshooting
- If `CUDA available: False`, a free GPU was not assigned. Try again later or use CPU (slow).
- If dependency installation fails, the upstream SadTalker dependency versions may conflict with the current Colab image; this is an environment issue, not proof that lip-sync worked.
- If no MP4 appears, do not treat the run as successful. Inspect the inference traceback and verify checkpoints.
- SadTalker is an open-source model project, but compute, availability, and model hosting are not guaranteed to be free.